**UJI AKURASI KLASIFIKASI LAHAN TERBANGUN — DYNAMIC WORLD**

Provinsi D.I. Yogyakarta, tahun 2025

* Membandingkan klasifikasi kelas 'built' Dynamic World (rata-rata probabilitas tahunan, ambang 0,5) terhadap label rujukan hasil interpretasi visual citra resolusi tinggi Google Earth pada 200 titik sampel acak berstratifikasi.
* Uji ini berperan sebagai JUSTIFIKASI kelayakan data, bukan sebagai alat
koreksi. Nilai luas lahan terbangun yang dipakai dalam model sistem dinamis adalah luas peta apa adanya, yaitu 55.030 ha untuk tahun 2025.
* Metrik mengikuti praktik baku penilaian akurasi penginderaan jauh: Olofsson, P., Foody, G. M., Herold, M., Stehman, S. V., Woodcock, C. E., & Wulder, M. A. (2014). Good practices for estimating area and assessing accuracy of land change. Remote Sensing of Environment, 148, 42-57. https://doi.org/10.1016/j.rse.2014.02.015

**CATATAN PENTING TENTANG SAMPEL BERSTRATIFIKASI**

*  Sampel diambil 100 titik per kelas prediksi, bukan proporsional terhadap luas kelas di lapangan. Karena itu Overall Accuracy sederhana (rasio titik benar terhadap total titik) BIAS, sebab kelas minoritas terwakili berlebih.
*   Skrip ini menghitung dua versi:
1.   tak terbobot  - untuk pelaporan deskriptif
2.   terbobot luas - estimator tak bias, sesuai Olofsson et al. (2014)

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import numpy as np
import pandas as pd
from scipy import stats

**KONFIGURASI**

In [24]:
PATH_LABEL = "/content/drive/MyDrive/Skripsi_Citra/Uji Akurasi Lokal/LembarKerja_UjiAkurasi_DW_2025.xlsx"
PATH_KUNCI = "/content/drive/MyDrive/Skripsi_Citra/Uji Akurasi Lokal/KunciModel_DW_2025.csv"

# Luas kelas hasil pemetaan Dynamic World 2025 (hektar)
# dipakai untuk pembobotan proporsi luas strata
LUAS_TERBANGUN = 55_029.5      # luas kelas built, peta biner ambang prob >= 0,5
LUAS_TOTAL     = 315_871.1     # cakupan wilayah terpetakan
LUAS_BUKAN     = LUAS_TOTAL - LUAS_TERBANGUN

NAMA = {0: "Bukan terbangun", 1: "Terbangun"}

1. MENGGABUNGKAN LABEL RUJUKAN DENGAN PREDIKSI MODEL

In [25]:
label = pd.read_excel(PATH_LABEL)
kunci = pd.read_csv(PATH_KUNCI)

for d in (label, kunci):
    d["nomor"] = d["nomor"].astype(int)

df = (label[["nomor", "kelas_referensi", "tgl_citra", "catatan"]]
      .merge(kunci[["nomor", "kelas_dw", "prob"]], on="nomor", how="inner"))
df["kelas_referensi"] = df["kelas_referensi"].astype(int)
df["kelas_dw"] = df["kelas_dw"].astype(int)

print("=" * 74)
print("UJI AKURASI KLASIFIKASI LAHAN TERBANGUN DYNAMIC WORLD 2025")
print("=" * 74)
print(f"Titik sampel tergabung : {len(df)}")
print(f"Prediksi model         : "
      f"{(df.kelas_dw==1).sum()} terbangun, {(df.kelas_dw==0).sum()} bukan")
print(f"Rujukan visual         : "
      f"{(df.kelas_referensi==1).sum()} terbangun, "
      f"{(df.kelas_referensi==0).sum()} bukan")

UJI AKURASI KLASIFIKASI LAHAN TERBANGUN DYNAMIC WORLD 2025
Titik sampel tergabung : 200
Prediksi model         : 100 terbangun, 100 bukan
Rujukan visual         : 71 terbangun, 129 bukan


2. MATRIKS KONFUSI

In [26]:
# baris = prediksi model (peta), kolom = rujukan lapangan
mk = pd.crosstab(df["kelas_dw"], df["kelas_referensi"],
                 rownames=["Peta"], colnames=["Rujukan"])
mk = mk.reindex(index=[0, 1], columns=[0, 1], fill_value=0)

n11 = int(mk.loc[1, 1])   # peta terbangun, rujukan terbangun  (benar)
n10 = int(mk.loc[1, 0])   # peta terbangun, rujukan bukan      (komisi)
n01 = int(mk.loc[0, 1])   # peta bukan, rujukan terbangun      (omisi)
n00 = int(mk.loc[0, 0])   # peta bukan, rujukan bukan          (benar)
n = n11 + n10 + n01 + n00

print()
print("-" * 74)
print("MATRIKS KONFUSI (jumlah titik)")
print("-" * 74)
tampil = mk.rename(index=NAMA, columns=NAMA)
tampil["Total"] = tampil.sum(axis=1)
tampil.loc["Total"] = tampil.sum()
print(tampil.to_string())


--------------------------------------------------------------------------
MATRIKS KONFUSI (jumlah titik)
--------------------------------------------------------------------------
Rujukan          Bukan terbangun  Terbangun  Total
Peta                                              
Bukan terbangun               95          5    100
Terbangun                     34         66    100
Total                        129         71    200


3. AKURASI TAK TERBOBOT

In [27]:
def ik_wilson(k, m, konf=0.95):
    """Selang kepercayaan Wilson untuk proporsi — lebih andal daripada
    pendekatan normal ketika proporsi mendekati 0 atau 1."""
    if m == 0:
        return (np.nan, np.nan)
    zz = stats.norm.ppf(1 - (1 - konf) / 2)
    pp = k / m
    d = 1 + zz**2 / m
    pusat = (pp + zz**2 / (2 * m)) / d
    lebar = zz * np.sqrt(pp * (1 - pp) / m + zz**2 / (4 * m**2)) / d
    return (max(0, pusat - lebar), min(1, pusat + lebar))

oa  = (n11 + n00) / n
ua1 = n11 / (n11 + n10)          # User's Accuracy kelas terbangun
ua0 = n00 / (n00 + n01)          # User's Accuracy kelas bukan
pa1 = n11 / (n11 + n01)          # Producer's Accuracy kelas terbangun
pa0 = n00 / (n00 + n10)          # Producer's Accuracy kelas bukan

print()
print("-" * 74)
print("AKURASI TAK TERBOBOT (deskriptif)")
print("-" * 74)
for nama, nilai, k, m in [
    ("Overall Accuracy",                    oa,  n11 + n00, n),
    ("User's Acc. - Terbangun",             ua1, n11, n11 + n10),
    ("User's Acc. - Bukan terbangun",       ua0, n00, n00 + n01),
    ("Producer's Acc. - Terbangun",         pa1, n11, n11 + n01),
    ("Producer's Acc. - Bukan terbangun",   pa0, n00, n00 + n10),
]:
    lo, hi = ik_wilson(k, m)
    print(f"  {nama:<34} {nilai*100:6.2f}%   IK95% [{lo*100:5.2f}%, {hi*100:5.2f}%]")

f1 = 2 * ua1 * pa1 / (ua1 + pa1) if (ua1 + pa1) > 0 else 0
print(f"  {'F1-score kelas terbangun':<34} {f1*100:6.2f}%")


--------------------------------------------------------------------------
AKURASI TAK TERBOBOT (deskriptif)
--------------------------------------------------------------------------
  Overall Accuracy                    80.50%   IK95% [74.46%, 85.39%]
  User's Acc. - Terbangun             66.00%   IK95% [56.28%, 74.54%]
  User's Acc. - Bukan terbangun       95.00%   IK95% [88.82%, 97.85%]
  Producer's Acc. - Terbangun         92.96%   IK95% [84.55%, 96.95%]
  Producer's Acc. - Bukan terbangun   73.64%   IK95% [65.44%, 80.48%]
  F1-score kelas terbangun            77.19%


4. KOEFISIEN KAPPA

In [28]:
po = oa
pe = (((n11 + n10) * (n11 + n01)) + ((n00 + n01) * (n00 + n10))) / n**2
kappa = (po - pe) / (1 - pe)

# ragam Kappa (Cohen, 1960; Fleiss et al., 1969)
se_k = np.sqrt((po * (1 - po)) / (n * (1 - pe) ** 2))
z = stats.norm.ppf(0.975)

def tafsir_kappa(k):
    if k < 0.00: return "buruk"
    if k < 0.20: return "sangat lemah"
    if k < 0.40: return "lemah"
    if k < 0.60: return "sedang"
    if k < 0.80: return "kuat"
    return "sangat kuat"

print()
print("-" * 74)
print("KOEFISIEN KAPPA")
print("-" * 74)
print(f"  Kappa                  = {kappa:.4f}")
print(f"  Galat baku             = {se_k:.4f}")
print(f"  IK 95%                 = [{kappa - z*se_k:.4f}, {kappa + z*se_k:.4f}]")
print(f"  Tafsiran (Landis & Koch 1977) = {tafsir_kappa(kappa)}")


--------------------------------------------------------------------------
KOEFISIEN KAPPA
--------------------------------------------------------------------------
  Kappa                  = 0.6100
  Galat baku             = 0.0560
  IK 95%                 = [0.5002, 0.7198]
  Tafsiran (Landis & Koch 1977) = kuat


5. ESTIMATOR TERBOBOT LUAS (Olofsson et al. 2014)

In [29]:
# Karena sampel berstratifikasi menurut kelas PETA, setiap strata diberi
# bobot sesuai proporsi luasnya di peta.
W = {1: LUAS_TERBANGUN / LUAS_TOTAL,
     0: LUAS_BUKAN / LUAS_TOTAL}
n_strata = {1: n11 + n10, 0: n00 + n01}

# matriks proporsi terbobot p[i][j]
p = {
    (1, 1): W[1] * n11 / n_strata[1],
    (1, 0): W[1] * n10 / n_strata[1],
    (0, 1): W[0] * n01 / n_strata[0],
    (0, 0): W[0] * n00 / n_strata[0],
}

oa_w  = p[(1, 1)] + p[(0, 0)]
ua1_w = n11 / n_strata[1]        # sama dengan versi tak terbobot
ua0_w = n00 / n_strata[0]
pa1_w = p[(1, 1)] / (p[(1, 1)] + p[(0, 1)])
pa0_w = p[(0, 0)] / (p[(0, 0)] + p[(1, 0)])

# --- galat baku Overall Accuracy terbobot ---
var_oa = sum(
    W[i] ** 2 *
    ((n11 if i == 1 else n00) / n_strata[i]) *
    (1 - (n11 if i == 1 else n00) / n_strata[i]) / (n_strata[i] - 1)
    for i in (0, 1)
)
se_oa = np.sqrt(var_oa)

# --- galat baku User's Accuracy (Olofsson et al. 2014, pers. 6) ---
#     V(U_i) = U_i (1 - U_i) / (n_i. - 1)
def se_ua(u, n_i):
    return np.sqrt(u * (1 - u) / (n_i - 1))

se_ua1 = se_ua(ua1_w, n_strata[1])
se_ua0 = se_ua(ua0_w, n_strata[0])

# --- galat baku Producer's Accuracy (pers. 7) ---
#     memperhitungkan ragam dari strata peta lain yang menyumbang
#     ke kelas rujukan yang sama
N = {1: LUAS_TERBANGUN, 0: LUAS_BUKAN}
nij = {(1, 1): n11, (1, 0): n10, (0, 1): n01, (0, 0): n00}

def se_pa(j):
    """Galat baku Producer's Accuracy kelas j."""
    N_dot_j = sum(N[i] * nij[(i, j)] / n_strata[i] for i in (0, 1))
    U_j = nij[(j, j)] / n_strata[j]
    P_j = N[j] * U_j / N_dot_j

    suku1 = N[j] ** 2 * (1 - P_j) ** 2 * U_j * (1 - U_j) / (n_strata[j] - 1)
    suku2 = P_j ** 2 * sum(
        N[i] ** 2 * (nij[(i, j)] / n_strata[i]) *
        (1 - nij[(i, j)] / n_strata[i]) / (n_strata[i] - 1)
        for i in (0, 1) if i != j
    )
    return np.sqrt((suku1 + suku2) / N_dot_j ** 2)

se_pa1 = se_pa(1)
se_pa0 = se_pa(0)

print()
print("-" * 74)
print("ESTIMATOR TERBOBOT LUAS (tak bias, Olofsson et al. 2014)")
print("-" * 74)
print(f"  Bobot strata: terbangun {W[1]:.4f} | bukan terbangun {W[0]:.4f}")
print()
print(f"{'  Metrik':<36}{'Nilai':>9}{'IK 95%':>24}")
print("-" * 74)
for nama, nilai, se in [
    ("Overall Accuracy",                  oa_w,  se_oa),
    ("User's Acc. - Terbangun",           ua1_w, se_ua1),
    ("User's Acc. - Bukan terbangun",     ua0_w, se_ua0),
    ("Producer's Acc. - Terbangun",       pa1_w, se_pa1),
    ("Producer's Acc. - Bukan terbangun", pa0_w, se_pa0),
]:
    lo, hi = max(0, nilai - z*se), min(1, nilai + z*se)
    print(f"  {nama:<34}{nilai*100:>8.2f}%   "
          f"[{lo*100:>6.2f}%, {hi*100:>6.2f}%]")


--------------------------------------------------------------------------
ESTIMATOR TERBOBOT LUAS (tak bias, Olofsson et al. 2014)
--------------------------------------------------------------------------
  Bobot strata: terbangun 0.1742 | bukan terbangun 0.8258

  Metrik                                Nilai                  IK 95%
--------------------------------------------------------------------------
  Overall Accuracy                     89.95%   [ 86.05%,  93.85%]
  User's Acc. - Terbangun              66.00%   [ 56.67%,  75.33%]
  User's Acc. - Bukan terbangun        95.00%   [ 90.71%,  99.29%]
  Producer's Acc. - Terbangun          73.58%   [ 56.66%,  90.50%]
  Producer's Acc. - Bukan terbangun    92.98%   [ 91.16%,  94.80%]


6. DIAGNOSTIK KESALAHAN

In [30]:
print()
print("-" * 74)
print("DIAGNOSTIK KESALAHAN KLASIFIKASI")
print("-" * 74)
salah = df[df.kelas_dw != df.kelas_referensi]
print(f"  Total titik salah klasifikasi : {len(salah)} dari {n} "
      f"({len(salah)/n*100:.1f}%)")
print(f"    komisi  (peta terbangun, rujukan bukan) : {n10}")
print(f"    omisi   (peta bukan, rujukan terbangun) : {n01}")

print()
print("  Sebaran probabilitas 'built' menurut ketepatan:")
for kel, grup in df.groupby(df.kelas_dw == df.kelas_referensi):
    lbl = "benar" if kel else "salah"
    print(f"    {lbl:<6} n={len(grup):>3}  rata-rata prob = {grup['prob'].mean():.3f}  "
          f"median = {grup['prob'].median():.3f}")

print()
print("  Ketepatan menurut selang probabilitas:")
tepi = [0, 0.25, 0.40, 0.50, 0.60, 0.75, 1.0]
df["selang"] = pd.cut(df["prob"], bins=tepi, include_lowest=True)
ring_prob = df.groupby("selang", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "benar": (g.kelas_dw == g.kelas_referensi).sum(),
        "akurasi_%": (g.kelas_dw == g.kelas_referensi).mean() * 100,
    }), include_groups=False)
print(ring_prob.round(1).to_string())


--------------------------------------------------------------------------
DIAGNOSTIK KESALAHAN KLASIFIKASI
--------------------------------------------------------------------------
  Total titik salah klasifikasi : 39 dari 200 (19.5%)
    komisi  (peta terbangun, rujukan bukan) : 34
    omisi   (peta bukan, rujukan terbangun) : 5

  Sebaran probabilitas 'built' menurut ketepatan:
    salah  n= 39  rata-rata prob = 0.574  median = 0.606
    benar  n=161  rata-rata prob = 0.330  median = 0.172

  Ketepatan menurut selang probabilitas:
                   n  benar  akurasi_%
selang                                
(-0.001, 0.25]  88.0   86.0       97.7
(0.25, 0.4]      6.0    5.0       83.3
(0.4, 0.5]       6.0    4.0       66.7
(0.5, 0.6]      25.0   13.0       52.0
(0.6, 0.75]     74.0   52.0       70.3
(0.75, 1.0]      1.0    1.0      100.0


7. TABEL SIAP PAKAI UNTUK BAB IV

In [31]:
def fmt(nilai, se):
    lo, hi = max(0, nilai - z*se), min(1, nilai + z*se)
    return f"{nilai*100:.2f}%", f"[{lo*100:.2f}%, {hi*100:.2f}%]"

ringkasan = pd.DataFrame({
    "Metrik": [
        "Jumlah titik sampel",
        "Overall Accuracy (tak terbobot)",
        "Overall Accuracy (terbobot luas)",
        "User's Accuracy - Terbangun",
        "User's Accuracy - Bukan terbangun",
        "Producer's Accuracy - Terbangun",
        "Producer's Accuracy - Bukan terbangun",
        "F1-score kelas terbangun",
        "Koefisien Kappa",
    ],
    "Nilai": [
        f"{n}",
        f"{oa*100:.2f}%",
        fmt(oa_w,  se_oa)[0],
        fmt(ua1_w, se_ua1)[0],
        fmt(ua0_w, se_ua0)[0],
        fmt(pa1_w, se_pa1)[0],
        fmt(pa0_w, se_pa0)[0],
        f"{f1*100:.2f}%",
        f"{kappa:.4f}",
    ],
    "IK 95%": [
        "-",
        f"[{ik_wilson(n11+n00, n)[0]*100:.2f}%, {ik_wilson(n11+n00, n)[1]*100:.2f}%]",
        fmt(oa_w,  se_oa)[1],
        fmt(ua1_w, se_ua1)[1],
        fmt(ua0_w, se_ua0)[1],
        fmt(pa1_w, se_pa1)[1],
        fmt(pa0_w, se_pa0)[1],
        "-",
        f"[{kappa - z*se_k:.4f}, {kappa + z*se_k:.4f}]",
    ],
})

print()
print("=" * 74)
print("RINGKASAN UNTUK BAB IV")
print("=" * 74)
print(ringkasan.to_string(index=False))

# Define the base path for saving to Google Drive
DRIVE_SAVE_PATH = "/content/drive/MyDrive/Skripsi_Citra/Uji Akurasi Lokal"

ringkasan.to_csv(f"{DRIVE_SAVE_PATH}/Hasil_UjiAkurasi_DW_2025.csv", index=False)
mk.rename(index=NAMA, columns=NAMA).to_csv(f"{DRIVE_SAVE_PATH}/Matriks_Konfusi_DW_2025.csv")
salah.to_csv(f"{DRIVE_SAVE_PATH}/Titik_SalahKlasifikasi_DW_2025.csv", index=False)

print()
print("Berkas keluaran:")
print(f"  {DRIVE_SAVE_PATH}/Hasil_UjiAkurasi_DW_2025.csv")
print(f"  {DRIVE_SAVE_PATH}/Matriks_Konfusi_DW_2025.csv")
print(f"  {DRIVE_SAVE_PATH}/Titik_SalahKlasifikasi_DW_2025.csv")


RINGKASAN UNTUK BAB IV
                               Metrik  Nilai           IK 95%
                  Jumlah titik sampel    200                -
      Overall Accuracy (tak terbobot) 80.50% [74.46%, 85.39%]
     Overall Accuracy (terbobot luas) 89.95% [86.05%, 93.85%]
          User's Accuracy - Terbangun 66.00% [56.67%, 75.33%]
    User's Accuracy - Bukan terbangun 95.00% [90.71%, 99.29%]
      Producer's Accuracy - Terbangun 73.58% [56.66%, 90.50%]
Producer's Accuracy - Bukan terbangun 92.98% [91.16%, 94.80%]
             F1-score kelas terbangun 77.19%                -
                      Koefisien Kappa 0.6100 [0.5002, 0.7198]

Berkas keluaran:
  /content/drive/MyDrive/Skripsi_Citra/Uji Akurasi Lokal/Hasil_UjiAkurasi_DW_2025.csv
  /content/drive/MyDrive/Skripsi_Citra/Uji Akurasi Lokal/Matriks_Konfusi_DW_2025.csv
  /content/drive/MyDrive/Skripsi_Citra/Uji Akurasi Lokal/Titik_SalahKlasifikasi_DW_2025.csv
